# QR3.4 — Regime overlay → A5 risk-aversion λ

The HMM (QR3.2, filtered/causal) plus the anti-whipsaw debounce (QR3.3) produce a
**committed market regime** (0 = calm, 1 = elevated, 2 = turbulent, sorted by
realized vol). QR3.4 maps that regime to the A5 mean-variance **λ**: a more
turbulent regime selects a larger λ, so `PortfolioBuilder`'s `−λ/2·wᵀΣw` term
pulls the book toward a minimum-variance / lower-gross posture. The map lives in
`config/regime_lambda.yaml` and is consumed in C++ by `qse::RegimeLambda`
(verified by `RegimeLambdaTest`: the turbulent regime provably lowers portfolio
variance and gross exposure).

This notebook plots the **SPY equity curve colored by the committed HMM state**,
so you can see the overlay de-risks (large λ) exactly during drawdowns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REGIME_LAMBDA = {0: 0.0, 1: 5.0, 2: 50.0}   # config/regime_lambda.yaml
LABELS = {0: "calm", 1: "elevated", 2: "turbulent"}
COLORS = {0: "tab:green", 1: "tab:orange", 2: "tab:red"}

## Load the market proxy and the committed regime

`data/regime/SPY.csv` is the market proxy (fetched via the same Alpaca IEX path
as the QR4 universe); `data/regime/regime_committed.parquet` carries the
debounced `committed_state` (QR3.3). Regenerate both with
`regime_features.py → regime_hmm.py → regime_debounce.py`.

In [ ]:
spy = pd.read_csv("../../../data/regime/SPY.csv", index_col=0, parse_dates=True).sort_index()
equity = spy["close"] / spy["close"].iloc[0]          # total-return index, start = 1.0

reg = pd.read_parquet("../../../data/regime/regime_committed.parquet")["committed_state"]
eq = equity.reindex(reg.index)                        # align to the regime span
print(f"{len(reg)} regime days, {reg.index[0].date()} -> {reg.index[-1].date()}")

## SPY equity curve, colored by committed regime

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for s in (0, 1, 2):
    seg = eq.where(reg == s)
    ax.scatter(seg.index, seg, s=8, c=COLORS[s], label=f"{s}: {LABELS[s]} (λ={REGIME_LAMBDA[s]:.0f})")
ax.plot(eq.index, eq, lw=0.4, color="gray", alpha=0.4, zorder=0)
ax.set_ylabel("SPY total return (start = 1.0)"); ax.set_xlabel("date")
ax.set_title("SPY equity curve colored by committed HMM regime (-> A5 lambda)")
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

## Regime occupancy and the λ the overlay would apply

Turbulent days (largest λ) cluster in the drawdowns — the 2022 bear bottom and
the April 2025 selloff — which is exactly where the overlay pushes the book
toward minimum variance.

In [ ]:
occ = reg.value_counts(normalize=True).sort_index()
summary = pd.DataFrame({
    "days": reg.value_counts().sort_index(),
    "occupancy": occ.round(3),
    "lambda": pd.Series(REGIME_LAMBDA),
    "regime": pd.Series(LABELS),
})
summary

## The C++ integration (what actually moves λ)

In the engine, `qse::RegimeLambda::apply(base_config, committed_state)` returns a
`PortfolioBuilder` config with `risk_aversion = regime_lambda[state]`.
`RegimeLambdaTest` proves the done-when: switching the injected regime from calm
(λ=0) to turbulent (λ=50) **lowers the optimized portfolio's variance** and,
on a factor-exposed book, **scales its gross exposure down** — the overlay is
risk control, not alpha.